# 미션 16 — 모델 추론 & 검증 (`inference.ipynb`)

`modeling.ipynb`에서 저장한 **ONNX 파일을 기반으로** 세 가지 모델(FP32 `.pth`, INT8 `.pth`, ONNX)을
MNIST **평가 데이터셋 전체(10,000장)** 로 검증한다.

검증 항목: (1) 전체 정확도 (2) 단일 이미지 예측 일치 (3) 워밍업+반복 속도 측정(P50/P90/P99)

> 학습 자료 매핑 — ONNX Runtime 추론(Ch.2), 데이터셋 전체 정확도(Ch.7), 올바른 속도 측정(Ch.6)

## 0. 라이브러리 설치

In [2]:
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 90.4 MB/s eta 0:00:00


In [10]:
import os, time, warnings
import numpy as np, torch, torch.nn as nn
import torch.ao.quantization as tq
import onnxruntime as ort
warnings.filterwarnings("ignore")
torch.backends.quantized.engine = "fbgemm"
MODELDIR = "outputs"   # modeling.ipynb 로컬 PC 저장 위치
MODELDIR = "/content/drive/MyDrive/codeit/mission16/outputs"   # modeling.ipynb 구글 코랩(Colab) 저장 위치

## Google Drive 마운트 (선택)
Colab 환경에서 데이터 파일을 Drive에서 불러올 경우 사용.

In [8]:
# Colab에서 Google Drive 마운트 (옵션)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
    print("로컬 환경에서 실행 중입니다.")

Mounted at /content/drive


In [9]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 드라이브 마운트 작업 후 아래 경로에서 폰트 가져오기
# /content/drive/MyDrive/fonts/NanumGothic.ttf

font_path = f'/content/drive/MyDrive/fonts/NanumGothic.ttf'  # 설치한 폰트 경로
fm.fontManager.addfont(font_path)  # 폰트 경로 추가

plt.rcParams['font.family'] = 'NanumGothic'  # 사용 폰트 입력
plt.rcParams['axes.unicode_minus'] = False   # 음수 부호 사용

## 1. 평가 데이터셋 준비

In [4]:
# MNIST 로더: torchvision 기본 미러가 막힌 환경에서도 동작하도록 fallback 포함
import os, gzip, pickle, urllib.request
import numpy as np, torch
from torch.utils.data import TensorDataset, DataLoader

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

def _load_from_pkl_mirror():
    url = "https://raw.githubusercontent.com/mnielsen/neural-networks-and-deep-learning/master/data/mnist.pkl.gz"
    path = "mnist.pkl.gz"
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    with gzip.open(path, "rb") as f:
        tr, va, te = pickle.load(f, encoding="latin1")
    def pack(X, y):
        X = ((X.reshape(-1,1,28,28).astype("float32")) - MNIST_MEAN) / MNIST_STD
        return torch.from_numpy(X), torch.from_numpy(y.astype("int64"))
    Xtr, ytr = pack(np.concatenate([tr[0],va[0]]), np.concatenate([tr[1],va[1]]))
    Xte, yte = pack(te[0], te[1])
    return TensorDataset(Xtr,ytr), TensorDataset(Xte,yte)

def get_mnist_loaders(batch_size=128, test_batch=256):
    """가능하면 torchvision.datasets.MNIST 사용, 실패 시 github 미러로 fallback."""
    try:
        import torchvision, torchvision.transforms as T
        tf = T.Compose([T.ToTensor(), T.Normalize((MNIST_MEAN,), (MNIST_STD,))])
        train_ds = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=tf)
        test_ds  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tf)
        print("데이터 출처: torchvision.datasets.MNIST")
    except Exception as e:
        print("torchvision MNIST 다운로드 실패 → github 미러 fallback:", type(e).__name__)
        train_ds, test_ds = _load_from_pkl_mirror()
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=test_batch, shuffle=False)
    return train_loader, test_loader

In [5]:
_, test_loader = get_mnist_loaders()
print("평가 데이터 수:", len(test_loader.dataset), "장")

100%|██████████| 9.91M/9.91M [00:00<00:00, 15.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 408kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.79MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.42MB/s]

데이터 출처: torchvision.datasets.MNIST
평가 데이터 수: 10000 장


## 2. 모델 클래스 정의 (FP32 `state_dict` 로드용)
INT8 모델은 TorchScript라 클래스가 필요 없지만, FP32 `state_dict` 복원에는 구조 정의가 필요합니다.

In [6]:
import torch.nn as nn
import torch.ao.quantization as tq

class SimpleCNN(nn.Module):
    """MNIST용 소형 CNN. Static PTQ 양자화를 위해 QuantStub/DeQuantStub 포함.
       (Conv+ReLU를 모듈로 분리해두면 fuse_modules로 융합 가능)"""
    def __init__(self):
        super().__init__()
        self.quant = tq.QuantStub()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1); self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1); self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)
        self.fc    = nn.Linear(32*7*7, 10)
        self.dequant = tq.DeQuantStub()
    def forward(self, x):
        x = self.quant(x)
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = torch.flatten(x, 1)   # 주의: .view() 대신 flatten (양자화 텐서 호환)
        x = self.fc(x)
        return self.dequant(x)

## 3. ONNX 추론 — 전체 정확도 (핵심)
`onnxruntime`으로 ONNX 모델을 로드하고 테스트셋 전체 정확도를 계산합니다. ONNX Runtime은 numpy 배열을 입력으로 받습니다(Ch.2).

In [11]:
sess = ort.InferenceSession(f"{MODELDIR}/mission_16_simplecnn.onnx",
                            providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
print("실제 사용 provider:", sess.get_providers())   # fallback 여부 확인 (Ch.6)

def evaluate_onnx(session, loader):
    correct = total = 0
    for xb, yb in loader:
        out = session.run(None, {input_name: xb.numpy()})[0]
        correct += (out.argmax(1) == yb.numpy()).sum(); total += yb.size(0)
    return correct / total * 100

acc_onnx = evaluate_onnx(sess, test_loader)
print(f"[ONNX] 전체 정확도: {acc_onnx:.2f}%")

실제 사용 provider: ['CPUExecutionProvider']
[ONNX] 전체 정확도: 98.24%


## 4. FP32 / INT8 `.pth` 로드 및 정확도
- FP32: 구조 생성 후 `load_state_dict`
- INT8: `torch.jit.load` (TorchScript — 클래스 불필요)

In [12]:
model_fp32 = SimpleCNN()
model_fp32.load_state_dict(torch.load(f"{MODELDIR}/mission_16_simplecnn_fp32.pth"))
model_fp32.eval()

model_int8 = torch.jit.load(f"{MODELDIR}/mission_16_simplecnn_int8.pth")
model_int8.eval()

def evaluate_torch(model, loader):
    correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            correct += (model(xb).argmax(1) == yb).sum().item(); total += yb.size(0)
    return correct / total * 100

acc_fp32 = evaluate_torch(model_fp32, test_loader)
acc_int8 = evaluate_torch(model_int8, test_loader)
print(f"[FP32 .pth] 정확도: {acc_fp32:.2f}%")
print(f"[INT8 .pth] 정확도: {acc_int8:.2f}%  | 손실 {acc_fp32-acc_int8:.2f}%p")

[FP32 .pth] 정확도: 98.24%
[INT8 .pth] 정확도: 98.16%  | 손실 0.08%p


## 5. 단일 이미지 예측 일치 확인
세 방식(FP32 / INT8 / ONNX)이 같은 이미지에 대해 동일한 클래스를 예측하는지 확인합니다.
동일하면 변환 파이프라인 전체가 정확히 작동한다는 뜻입니다(Ch.3·Ch.7).

In [13]:
xb, yb = next(iter(test_loader))
x1 = xb[:1]
p_onnx = int(sess.run(None, {input_name: x1.numpy()})[0].argmax(1)[0])
with torch.no_grad():
    p_fp32 = int(model_fp32(x1).argmax(1)[0])
    p_int8 = int(model_int8(x1).argmax(1)[0])
print(f"FP32={p_fp32}  INT8={p_int8}  ONNX={p_onnx}  (정답={int(yb[0])})")
print("세 예측 일치:", p_fp32 == p_int8 == p_onnx)

FP32=7  INT8=7  ONNX=7  (정답=7)
세 예측 일치: True


## 6. 속도 비교 — 워밍업 + 반복 측정 (Ch.6)
1회 측정은 초기화 오버헤드에 흔들리므로, 워밍업 10회 후 50회 반복하여 평균·표준편차·P50/P90/P99를 계산합니다.

In [14]:
def benchmark(run_fn, n_warmup=10, n_runs=50):
    for _ in range(n_warmup):       # 워밍업(측정 제외)
        run_fn()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter(); run_fn(); times.append((time.perf_counter()-t0)*1000)
    t = np.array(times)
    return {"mean": t.mean(), "std": t.std(),
            "p50": np.percentile(t,50), "p90": np.percentile(t,90), "p99": np.percentile(t,99)}

x1_t = xb[:1]; x1_np = x1_t.numpy()
r_fp32 = benchmark(lambda: model_fp32(x1_t))
r_int8 = benchmark(lambda: model_int8(x1_t))
r_onnx = benchmark(lambda: sess.run(None, {input_name: x1_np}))
for name, r in [("PyTorch FP32", r_fp32), ("PyTorch INT8", r_int8), ("ONNX Runtime", r_onnx)]:
    print(f"[{name:13}] mean {r['mean']:.3f}±{r['std']:.3f}ms | "
          f"P50 {r['p50']:.3f} P90 {r['p90']:.3f} P99 {r['p99']:.3f}ms")

[PyTorch FP32 ] mean 0.677±0.054ms | P50 0.669 P90 0.733 P99 0.851ms
[PyTorch INT8 ] mean 0.452±0.275ms | P50 0.434 P90 0.566 P99 1.470ms
[ONNX Runtime ] mean 0.065±0.013ms | P50 0.061 P90 0.078 P99 0.113ms


## 7. 최종 비교표 (정확도 · 용량 · 속도)

In [15]:
import pandas as pd
def kb(fn): return round(os.path.getsize(f"{MODELDIR}/{fn}")/1024, 1)
summary = pd.DataFrame([
    {"방법":"PyTorch FP32 (.pth)", "정확도(%)":round(acc_fp32,2), "용량(KB)":kb("mission_16_simplecnn_fp32.pth"), "추론(ms)":round(r_fp32["mean"],3), "P99(ms)":round(r_fp32["p99"],3)},
    {"방법":"PyTorch INT8 (.pth)", "정확도(%)":round(acc_int8,2), "용량(KB)":kb("mission_16_simplecnn_int8.pth"), "추론(ms)":round(r_int8["mean"],3), "P99(ms)":round(r_int8["p99"],3)},
    {"방법":"ONNX Runtime (.onnx)","정확도(%)":round(acc_onnx,2), "용량(KB)":kb("mission_16_simplecnn.onnx"),     "추론(ms)":round(r_onnx["mean"],3), "P99(ms)":round(r_onnx["p99"],3)},
])
print(summary.to_string(index=False)); summary

                  방법  정확도(%)  용량(KB)  추론(ms)  P99(ms)
 PyTorch FP32 (.pth)   98.24    83.4   0.677    0.851
 PyTorch INT8 (.pth)   98.16    42.6   0.452    1.470
ONNX Runtime (.onnx)   98.24    81.3   0.065    0.113


,방법,정확도(%),용량(KB),추론(ms),P99(ms)
0,PyTorch FP32 (.pth),98.24,83.4,0.677,0.851
1,PyTorch INT8 (.pth),98.16,42.6,0.452,1.470
2,ONNX Runtime (.onnx),98.24,81.3,0.065,0.113


## 정리
- 세 포맷 모두 MNIST 10,000장 전체에서 정확도를 검증했고, 예측 클래스가 일치함 → 변환 파이프라인 정상.
- INT8 양자화로 용량이 줄었고 정확도 손실은 미미(≈0.1%p).
- 속도는 환경(CPU·batch=1)에 따라 달라질 수 있음(Ch.4·Ch.6). 공정 비교를 위해 워밍업+반복 측정을 적용함.

> INT8이 항상 빠른 것은 아니며(Ch.4.4), 실제 배포 환경에서 반드시 재측정해야 합니다.